# AXI test 47: three arms on ten tau2-bench retail tasks (Gemini 3.8 Flash)
Pre-registered page: `docs/TAU2_PILOT.md` (the plan, the gate, the predictions, all written before this run). Same model on both sides of the comparison: the benchmark's own agent (`B`), the same agent with the host's gate (`G`), and the host-held state with the same gate (`A`). The benchmark's own user simulator and evaluator score every run.

**What to know**
* **No GPU.** The free CPU runtime is enough. Runtime > Run all, top to bottom.
* **A key.** Put your key into Colab **Secrets** under the name `GEMINI_API_KEY` and switch **Notebook access** on (it is per notebook). The key is held in memory and is never printed or written to a file.
* **Cost and time.** Smoke test (task 0, three arms): a few cents, a few minutes. Full run (10 tasks, three arms): by token arithmetic under about $10 at list prices; a few tens of minutes.
* The pass counts are a first look (ten tasks, one trial). The token counts are the measurement.

### Cell 1: get this repository and the benchmark at the pinned commit (about 2 to 4 minutes)

In [ ]:
!rm -rf /content/axi /content/tau2-bench
!git clone -q --depth 1 https://github.com/gaurshalendra2412/axi-runtime /content/axi
!git clone -q https://github.com/sierra-research/tau2-bench /content/tau2-bench
!cd /content/tau2-bench && git checkout -q 768f3d8fe8528b4f32a5096f9ed008ae50f48ead && git log -1 --format='%H %cd'
!pip -q install -e /content/tau2-bench
import os
assert os.path.exists("/content/axi/python/axi/experiments/tau2_pilot_run.py"), "test 47 is not on GitHub yet"
print("ready")

### Cell 2: the key (Colab Secrets, name `GEMINI_API_KEY`)

In [ ]:
import os
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
print("key set:", bool(os.environ["GEMINI_API_KEY"]))

### Cell 3: the offline checks (no model call): the gate, the held state, the glue and the summary

In [ ]:
%cd /content/axi
!PYTHONPATH=python python3 python/tests/test_tau2_retail_core.py | tail -2
!PYTHONPATH=python python3 python/tests/test_tau2_retail_agents.py | tail -2
!PYTHONPATH=python python3 python/tests/test_tau2_pilot_run.py | tail -2

### Cell 4: smoke test, task 0 on all three arms (a few cents)
If this fails, stop and send the error text; do not run Cell 5.

In [ ]:
%cd /content/axi/python
!python3 -m axi.experiments.tau2_pilot_run run --arms B G A --tasks 0 --tag smoke --concurrency 1

In [ ]:
!python3 -m axi.experiments.tau2_pilot_run summary --tag smoke --tasks 0 2>&1 | tail -40

### Cell 5: the full run, ten tasks on three arms (tens of minutes)

In [ ]:
!python3 -m axi.experiments.tau2_pilot_run run --arms B G A --tag t47 --concurrency 2

### Cell 6: the tables and the eight readings of the plan page

In [ ]:
!python3 -m axi.experiments.tau2_pilot_run summary --tag t47 2>&1 | tail -60
import json
print(json.dumps(json.load(open("t47_summary.json"))))

In [ ]:
from google.colab import files
files.download("t47_summary.json")